# IMUSA: Multimodal Punjabi Meme Sentiment Analysis — Training & Inference Pipeline

This notebook trains a Late-Fusion Multimodal Model (**Vision Transformer + XLM-RoBERTa + Gated Fusion + Focal Loss**) for 4-class sentiment classification on Punjabi memes (`Sarcasm`, `Motivational`, `Neutral`, `Offensive`).

**Hardware**: Google Colab T4 GPU (Free tier)

## 1. Environment Setup & Repository Clone

In [1]:
!pip install -q uv
!git clone https://github.com/shubhojit-mitra-dev/imusa-multimodal-sentiment.git project
%cd project
!uv sync --all-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 74.6 MB/s eta 0:00:00
Cloning into 'project'...
remote: Enumerating objects: 431, done.
remote: Counting objects: 100% (431/431), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 431 (delta 191), reused 423 (delta 189), pack-reused 0 (from 0)
Receiving objects: 100% (431/431), 6.21 MiB | 19.21 MiB/s, done.
Resolving deltas: 100% (191/191), done.
/content/project
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 99 packages in 1ms
Prepared 96 packages in 58.66s
Installed 96 packages in 950ms
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.14.2
 + ast-serialize==0.8.0
 + certifi==2026.7.22
 + cfgv==3.5.0
 + click==8.4.2
 + contourpy==1.3.3
 + coverage==7.15.4
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.6.0
 + cuda-toolkit==13.0.3.0
 + cycler==0.12.1
 + distlib==0.4.3
 + filelock==3.32.2
 + fonttools==4.63.0
 + fsspec==2026.7.0
 + h11==0.16.0

## 2. Dataset Setup (Upload / Unzip data.zip)

Upload your `data.zip` file containing the `data/` directory.

In [2]:
from pathlib import Path

train_csv = Path("data/train/train_punjabi_dataset.csv")

if not train_csv.exists():
    print("Dataset not detected in project/data/")
    if Path("/content/data.zip").exists():
        print("Found /content/data.zip, extracting...")
        !unzip -q /content/data.zip -d /content/project/
    elif Path("data.zip").exists():
        print("Found data.zip in project directory, extracting...")
        !unzip -q data.zip -d /content/project/
    else:
        print("Please upload your 'data.zip' file:")
        from google.colab import files

        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".zip"):
                !unzip -q "{fname}" -d /content/project/

if train_csv.exists():
    print("Dataset successfully verified at data/train/train_punjabi_dataset.csv")
else:
    print("Dataset missing. Please ensure data.zip contains data/ directory structure.")

Dataset not detected in project/data/
Please upload your 'data.zip' file:


Saving data.zip to data.zip
Dataset successfully verified at data/train/train_punjabi_dataset.csv


## 3. Execute Dataset Cleaning & EDA Pipeline

In [3]:
!uv run python scripts/clean_data.py
!uv run python scripts/explore_data.py

2026-08-14 11:38:08,736 [INFO] imusa.data.cleaning: Starting dataset cleaning pipeline on /content/project/data/train/train_punjabi_dataset.csv
2026-08-14 11:38:08,839 [INFO] imusa.data.cleaning: Saved cleaned dataset (2891 rows) to /content/project/data/processed/train_clean.csv
     IMUSA Dataset Cleaning Report      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Pipeline Stage               ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Total Raw Rows Parsed        │  3002 │
│ Dropped (Missing Category)   │     0 │
│ Dropped (Invalid Category)   │     0 │
│ Dropped (Missing Image File) │     0 │
│ Dropped (Duplicates)         │   111 │
│ Final Clean Dataset Size     │  2891 │
└──────────────────────────────┴───────┘
2026-08-14 11:38:15,342 [INFO] imusa.data.explorer: Loaded cleaned dataset with 2891 samples for EDA
2026-08-14 11:38:16,349 [INFO] imusa.data.explorer: Saved class distribution plot to /content/project/outputs/exploration/class_distribution.png
2026-08-14 11:38:16,83

## 4. Train Multimodal Model with Cosine Warmup & Focal Loss

In [4]:
!uv run python scripts/train.py --epochs 10 --batch-size 16 --lr 2e-5 --loss focal --warmup-ratio 0.1

2026-08-14 11:38:36,306 [INFO] imusa.scripts.train: === IMUSA Multimodal Model Training Initialization ===
2026-08-14 11:38:36,306 [INFO] imusa.data.cleaning: Starting dataset cleaning pipeline on /content/project/data/train/train_punjabi_dataset.csv
2026-08-14 11:38:36,385 [INFO] imusa.data.cleaning: Saved cleaned dataset (2891 rows) to /content/project/data/processed/train_clean.csv
     IMUSA Dataset Cleaning Report      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Pipeline Stage               ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Total Raw Rows Parsed        │  3002 │
│ Dropped (Missing Category)   │     0 │
│ Dropped (Invalid Category)   │     0 │
│ Dropped (Missing Image File) │     0 │
│ Dropped (Duplicates)         │   111 │
│ Final Clean Dataset Size     │  2891 │
└──────────────────────────────┴───────┘
2026-08-14 11:38:41,033 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-08-14 11:38:41,279 [INFO] httpx: HTTP R

## 5. Run Test Set Inference & Generate Submission CSV

In [10]:
!uv run python scripts/predict.py --checkpoint outputs/checkpoints/best_model.pt --output outputs/submission.csv

2026-08-14 12:24:21,324 [INFO] __main__: Reading test dataset from /content/project/data/test/Test.csv...
2026-08-14 12:24:21,328 [INFO] __main__: Loaded 500 test samples.
2026-08-14 12:24:24,289 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-14 12:24:24,531 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-14 12:24:24,532 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-14 12:24:24,766 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-14 12:24:24,997 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base/tree/main/add

## 6. Download Results & Submission Artifacts

In [11]:
from google.colab import files

files.download("outputs/submission.csv")
files.download("outputs/checkpoints/best_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>